In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , precision_score , recall_score,f1_score,classification_report, confusion_matrix
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import matplotlib.pyplot as plt
import seaborn as sns


C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
mlflow.set_tracking_uri("https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow")


In [3]:
import dagshub
dagshub.init(repo_owner='Aayush10671', repo_name='yt-comment-sentiment-analysis', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as Aayush10671

Initialized MLflow to track repo "Aayush10671/yt-comment-sentiment-analysis"

Repository Aayush10671/yt-comment-sentiment-analysis initialized!

🏃 View run delicate-shark-48 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/0/runs/7007b21fca0c4159902b45c14b236243
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/0


In [4]:
df = pd.read_csv("preprocessed_data.csv")
df.shape

(36793, 2)

In [5]:
print(df['clean_comment'].isnull().sum())          # count of NaN
print(df['clean_comment'].dtype)                   # should be object or string
print(df['clean_comment'].apply(type).value_counts())  # see if all are strings
print(df['clean_comment'].str.len().value_counts().head()) # check empty strings

131
object
clean_comment
<class 'str'>      36662
<class 'float'>      131
Name: count, dtype: int64
clean_comment
14.0    439
15.0    429
24.0    421
19.0    420
22.0    416
Name: count, dtype: int64


In [6]:
# Drop rows where clean_comment is missing
df = df.dropna(subset=['clean_comment'])
# Remove rows where the comment is empty after stripping
df = df[df['clean_comment'].str.strip() != '']
# Ensure all values are strings (just in case)
df['clean_comment'] = df['clean_comment'].astype(str)

In [7]:
df = df.dropna(subset=['clean_comment'])
df = df[df['clean_comment'].str.strip() != '']
df['clean_comment'] = df['clean_comment'].astype(str)

In [8]:
mlflow.set_experiment("handeling imbalance data")

2026/07/25 22:43:43 INFO mlflow.tracking.fluent: Experiment with name 'handeling imbalance data' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/ecaa941f83a9432d8aeabfecba81f290', creation_time=1784999624282, experiment_id='5', last_update_time=1784999624282, lifecycle_stage='active', name='handeling imbalance data', tags={}, workspace='default'>

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import RandomOverSampler, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns


def run_imbalance_experiment(imbalance_method):

  
    ngram_range = (1, 3)
    max_features = 1000

    vectorizer = TfidfVectorizer(
        ngram_range=ngram_range,
        max_features=max_features
    )

    X = vectorizer.fit_transform(df["clean_comment"])
    y = df["category"].values

  
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
       
    )

    class_weight = None

 
    if imbalance_method == "class_weights":
        class_weight = "balanced"

    elif imbalance_method == "oversampling":
        sampler = RandomOverSampler(random_state=42)
        X_train, y_train = sampler.fit_resample(X_train, y_train)

    elif imbalance_method == "undersampling":
        sampler = RandomUnderSampler(random_state=42)
        X_train, y_train = sampler.fit_resample(X_train, y_train)

    elif imbalance_method == "adasyn":
        sampler = ADASYN(random_state=42)
        X_train, y_train = sampler.fit_resample(X_train, y_train)

    elif imbalance_method == "smoteenn":
        sampler = SMOTEENN(random_state=42)
        X_train, y_train = sampler.fit_resample(X_train, y_train)

    with mlflow.start_run():

        mlflow.set_tag("vectorizer", "TF-IDF")
        mlflow.set_tag("model", "RandomForest")
        mlflow.set_tag("imbalance_method", imbalance_method)

        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("max_features", max_features)
        mlflow.log_param("imbalance_method", imbalance_method)
        mlflow.log_param("n_estimators", 200)
        mlflow.log_param("max_depth", 15)

        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            class_weight=class_weight,
            random_state=42
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        # Accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Classification Report
        report = classification_report(
            y_test,
            y_pred,
            output_dict=True
        )

        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric_name, metric_value in metrics.items():
                    mlflow.log_metric(
                        f"{label}_{metric_name}",
                        metric_value
                    )

        # Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title("Confusion Matrix")

        plt.savefig("confusion_matrix.png")
        plt.close()

        mlflow.log_artifact("confusion_matrix.png")

        # Log Model
        mlflow.sklearn.log_model(
            sk_model=model,
            name=f"rf_{imbalance_method}"
        )

        print(f"{imbalance_method} -> Accuracy: {accuracy:.4f}")

In [11]:
methods = [
    "class_weights",
    "oversampling",
    "undersampling",
    "adasyn",
    "smoteenn"
]

for method in methods:
    run_imbalance_experiment(method)

2026/07/25 22:53:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


class_weights -> Accuracy: 0.6835
🏃 View run receptive-smelt-842 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/5/runs/621417a3eafe44d6b0c35d8b780915e3
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/5


2026/07/25 22:54:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


oversampling -> Accuracy: 0.6806
🏃 View run welcoming-toad-534 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/5/runs/f05e3006bdd144f19675f9c0149aab9f
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/5


2026/07/25 22:56:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


undersampling -> Accuracy: 0.6820
🏃 View run useful-grouse-229 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/5/runs/7ff10661fc67466c87f88d80004aabf3
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/5


2026/07/25 22:58:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


adasyn -> Accuracy: 0.6835
🏃 View run mysterious-goat-570 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/5/runs/0932e940200b4fb09dd97953ecd42754
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/5


2026/07/25 22:59:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


smoteenn -> Accuracy: 0.4440
🏃 View run defiant-owl-240 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/5/runs/21afcd0d027044b5ae766d376b884acc
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/5
